<a href="https://colab.research.google.com/github/neilm3006/ML-Driven-Thermal-Hotspot-Prediction-on-Jet-Turbine-Blades/blob/main/EDI_SY_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/airfoil_ml"
RAW = BASE + "/raw"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

nasa = pd.read_csv(f"{RAW}/NASA Airfoil Self-Noise Dataset/AirfoilSelfNoise.csv")
dlw  = pd.read_csv(f"{RAW}/DEEP LEAR WING/DeepLearWing.csv")
pg   = pd.read_csv(f"{RAW}/Airfoil Performance and Geometry Data/combinedAirfoilDataLabeled.csv")

print(nasa.shape)
print(dlw.shape)
print(pg.shape)


/tmp/ipykernel_15094/2840603609.py:5: DtypeWarning: Columns (70) have mixed types. Specify dtype option on import or set low_memory=False.
  pg   = pd.read_csv(f"{RAW}/Airfoil Performance and Geometry Data/combinedAirfoilDataLabeled.csv")


(1503, 6)
(819191, 8)
(867098, 71)


In [ ]:
nasa = nasa.dropna()

dlw = dlw.dropna()
pg  = pg.dropna()

nasa.reset_index(drop=True, inplace=True)
dlw.reset_index(drop=True, inplace=True)
pg.reset_index(drop=True, inplace=True)


In [ ]:
print("NASA COLUMNS:\n", nasa.columns)
print("\nDEEPLEARWING COLUMNS:\n", dlw.columns)
print("\nPERF + GEOM COLUMNS:\n", pg.columns)


NASA COLUMNS:
 Index(['f', 'alpha', 'c', 'U_infinity', 'delta', 'SSPL'], dtype='object')

DEEPLEARWING COLUMNS:
 Index(['name', 'angle', 'reynolds', 'x_coords', 'y_coords', 'cd', 'cl', 'cm'], dtype='object')

PERF + GEOM COLUMNS:
 Index(['airfoilName', 'upperSurfaceCoeff1', 'upperSurfaceCoeff2',
       'upperSurfaceCoeff3', 'upperSurfaceCoeff4', 'upperSurfaceCoeff5',
       'upperSurfaceCoeff6', 'upperSurfaceCoeff7', 'upperSurfaceCoeff8',
       'upperSurfaceCoeff9', 'upperSurfaceCoeff10', 'upperSurfaceCoeff11',
       'upperSurfaceCoeff12', 'upperSurfaceCoeff13', 'upperSurfaceCoeff14',
       'upperSurfaceCoeff15', 'upperSurfaceCoeff16', 'upperSurfaceCoeff17',
       'upperSurfaceCoeff18', 'upperSurfaceCoeff19', 'upperSurfaceCoeff20',
       'upperSurfaceCoeff21', 'upperSurfaceCoeff22', 'upperSurfaceCoeff23',
       'upperSurfaceCoeff24', 'upperSurfaceCoeff25', 'upperSurfaceCoeff26',
       'upperSurfaceCoeff27', 'upperSurfaceCoeff28', 'upperSurfaceCoeff29',
       'upperSurfaceCoeff3

In [ ]:
feature_cols = [c for c in pg.columns if 'Coeff' in c]
input_cols   = ['reynoldsNumber', 'alpha']
target_cols  = ['coefficientLift', 'coefficientDrag']

X_pg = pg[feature_cols + input_cols]
y_pg = pg[target_cols]


In [ ]:
nasa_inputs = nasa[['alpha', 'c', 'U_infinity']]
nasa_target = nasa['SSPL']


In [ ]:
dlw[['x_coords','y_coords']].head()


,x_coords,y_coords
0,1.0 0.95 0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.25 0.2 ...,0.0016 0.0124 0.0229 0.0428 0.061 0.0771 0.090...
1,1.0 0.95 0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.25 0.2 ...,0.0016 0.0124 0.0229 0.0428 0.061 0.0771 0.090...
2,1.0 0.95 0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.25 0.2 ...,0.0016 0.0124 0.0229 0.0428 0.061 0.0771 0.090...
3,1.0 0.95 0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.25 0.2 ...,0.0016 0.0124 0.0229 0.0428 0.061 0.0771 0.090...
4,1.0 0.95 0.9 0.8 0.7 0.6 0.5 0.4 0.3 0.25 0.2 ...,0.0016 0.0124 0.0229 0.0428 0.061 0.0771 0.090...


In [ ]:
# Convert space-separated strings into lists of floats
dlw['x_coords'] = dlw['x_coords'].apply(lambda s: [float(x) for x in s.split()])
dlw['y_coords'] = dlw['y_coords'].apply(lambda s: [float(x) for x in s.split()])


In [ ]:
print(dlw['x_coords'][0][:10])  # first 10 points of first airfoil
print(dlw['y_coords'][0][:10])


[1.0, 0.95, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.25]
[0.0016, 0.0124, 0.0229, 0.0428, 0.061, 0.0771, 0.0905, 0.1002, 0.1048, 0.1044]


In [ ]:
# Install xgboost if not already
!pip install xgboost --quiet

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_pg, y_pg, test_size=0.2, random_state=42)

# Create XGBoost regressor for multi-output
model = xgb.XGBRegressor(
    n_estimators=200,        # number of trees
    max_depth=6,             # tree depth
    learning_rate=0.1,       # step size shrinkage
    subsample=0.8,           # use 80% data per tree
    colsample_bytree=0.8,    # use 80% features per tree
    n_jobs=-1,               # use all cores
    objective='reg:squarederror',
    tree_method='hist'       # faster method for large datasets
)

# Fit model separately for lift and drag
models = {}
preds = {}
r2_scores = {}

for i, target in enumerate(['coefficientLift', 'coefficientDrag']):
    m = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        objective='reg:squarederror',
        tree_method='hist'
    )
    m.fit(X_train, y_train[target])
    p = m.predict(X_test)

    models[target] = m
    preds[target] = p
    r2_scores[target] = r2_score(y_test[target], p)

print("Lift R2:", r2_scores['coefficientLift'])
print("Drag R2:", r2_scores['coefficientDrag'])


In [ ]:
import numpy as np

def fitness_function(coeffs, model_lift, model_drag, reynolds, alpha):
    # coeffs: np.array of 62 coefficients
    input_vec = np.append(coeffs, [reynolds, alpha]).reshape(1, -1)

    cl = model_lift.predict(input_vec)[0]
    cd = model_drag.predict(input_vec)[0]

    # Objective: maximize L/D ratio
    ld_ratio = cl / (cd + 1e-6)

    # Penalize extreme coefficients (optional)
    penalty = np.sum(np.clip(np.abs(coeffs) - 0.5, 0, None)) * 0.01

    return ld_ratio - penalty


In [ ]:
!pip install deap --quiet



In [ ]:
import random
from deap import base, creator, tools

NUM_COEFF = 62  # 31 upper + 31 lower

# Create fitness and individual
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, -0.2, 0.2)  # coefficient range
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=NUM_COEFF)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Fitness evaluation
toolbox.register("evaluate", fitness_function, model_lift=models['coefficientLift'], model_drag=models['coefficientDrag'], reynolds=1e6, alpha=5.0)

# Genetic operators
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.05, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)


In [ ]:
POP_SIZE = 30
N_GEN = 20

pop = toolbox.population(n=POP_SIZE)

for gen in range(N_GEN):
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    # Crossover
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.8:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Mutation
    for mutant in offspring:
        if random.random() < 0.3:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Evaluate fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = [toolbox.evaluate(ind) for ind in invalid_ind]
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = (fit,)

    # Next generation
    pop[:] = offspring
    fits = [ind.fitness.values[0] for ind in pop]
    print(f"Gen {gen} max L/D: {max(fits)}")

# Get best individual
best_ind = tools.selBest(pop, 1)[0]
print("Best L/D:", best_ind.fitness.values[0])


In [ ]:
x = np.linspace(0.001, 0.999, 400)

from math import comb

def bern(i, n, x):
    return comb(n, i) * (x**i) * (1-x)**(n-i)

def CST(x, coeffs):
    N1, N2 = 0.5, 1.0
    n = len(coeffs)-1
    C = x**N1 * (1-x)**N2
    S = sum(coeffs[i] * bern(i, n, x) for i in range(len(coeffs)))
    return C * S

# split genome
best = tools.selBest(pop, 1)[0]
g = np.array(best)
N = len(g)//2
t = g[:N]
c = g[N:]

# thickness & camber
th = np.abs(CST(x, t))
cm = CST(x, c)*0.15   # reduce camber

# shape laws
th[-1] = 0
cm[-1] = 0
th[0] *= 2  # leading edge radius

# location lock (forces max thickness forward)
max_i = np.argmax(th)
target = int(0.3 * len(x))
shift = target - max_i
th = np.roll(th, shift)

# norma


In [ ]:
# NO waviness
if np.std(np.gradient(th,2)) > 0.5:
    th *= 0.5


In [ ]:
plt.figure(figsize=(9,3))
plt.plot(x, yu, 'b', lw=2)
plt.plot(x, yl, 'b', lw=2)
plt.fill_between(x, yl, yu, alpha=0.6)
plt.axis("equal")
plt.grid()
plt.title("Physically Constrained Airfoil")
plt.show()


In [ ]:
penalty = 0

# enforce max thickness location
if max_i < 0.15*len(x) or max_i > 0.45*len(x):
    penalty += 3

# enforce smooth curvature
curv = np.diff(np.sign(np.gradient(yu,2)))
penalty += np.sum(curv != 0)

fitness = predicted_lift - 0.02*predicted_drag - penalty
